# Lecture 9 — DMRG I: The Algorithm

---

## Overview

Exact diagonalisation is limited to roughly 40 sites. Lecture 08 showed how to *compress* an existing wavefunction into MPS form. But we still have not said how to *find* the ground state as an MPS in the first place.

This lecture introduces the **Density Matrix Renormalisation Group (DMRG)** — the algorithm that directly optimises an MPS ansatz to find the ground state without ever constructing the full wavefunction. DMRG was invented by Steven White in 1992 and is the most powerful method available for ground states of 1D quantum systems.

Lecture 10 covers the practical implementation using TeNPy.

---

In [ ]:
import numpy as np
import scipy.sparse as sp
from scipy.sparse.linalg import eigsh
import matplotlib.pyplot as plt

plt.rcParams.update({'font.size': 12, 'figure.dpi': 100})


def build_tfim(N, J=1.0, Gamma=1.0):
    dim = 2**N
    rows, cols, data = [], [], []
    for state in range(dim):
        diag = 0.0
        for i in range(N - 1):
            sz_i = 1 if (state >> i) & 1 else -1
            sz_j = 1 if (state >> (i + 1)) & 1 else -1
            diag -= J * sz_i * sz_j
        if diag != 0.0:
            rows.append(state); cols.append(state); data.append(diag)
        for i in range(N):
            rows.append(state); cols.append(state ^ (1 << i)); data.append(-Gamma)
    return sp.csr_matrix((data, (rows, cols)), shape=(dim, dim))

## 1. The Variational Principle for MPS

The foundation of DMRG is the variational principle: $E_0 = \min_{|\psi\rangle} \langle \psi | \hat{H} | \psi \rangle$.

We restrict this minimisation to MPS with fixed bond dimension $\chi$. For a given $\chi$, the MPS with the lowest energy is the best approximation to the true ground state within that class.

An MPS has $O(N\chi^2)$ free parameters. Direct gradient descent is possible but slow. DMRG exploits the MPS structure to make each step a small eigenvalue problem.

---

## 2. The DMRG Algorithm

**Key insight:** fix all tensors except one. The energy as a function of that tensor alone is a quadratic form — equivalent to finding the ground state of an **effective Hamiltonian** $H_{\text{eff}}$ of dimension $d\chi^2 \times d\chi^2$. Solving this with Lanczos costs $O(\chi^3)$ rather than $O(2^{2N})$.

**The sweep:** optimise left-to-right, then right-to-left, updating one (or two) tensors at each step. Each sweep reduces the energy. Terminate when the energy converges between sweeps.

---

In [ ]:
class SimpleDMRG:
    """
    Minimal single-site DMRG implementation for the TFIM.
    Illustrates the sweep mechanics and effective Hamiltonian construction.
    """

    def __init__(self, N, J, Gamma, chi_max):
        self.N = N
        self.J = J
        self.Gamma = Gamma
        self.chi_max = chi_max
        self.d = 2  # local Hilbert space dimension

        # Pauli matrices
        self.sz = np.array([[1, 0], [0, -1]], dtype=float)
        self.sx = np.array([[0, 1], [1, 0]], dtype=float)
        self.I2 = np.eye(2)

        # Initialise random MPS tensors (chi x d x chi)
        self.tensors = []
        chi_l = 1
        for i in range(N):
            chi_r = min(self.d**(i + 1), self.d**(N - i - 1), chi_max)
            A = np.random.randn(chi_l, self.d, chi_r)
            # Left-orthogonalise
            A = A.reshape(chi_l * self.d, chi_r)
            Q, _ = np.linalg.qr(A)
            chi_r_actual = Q.shape[1]
            self.tensors.append(Q.reshape(chi_l, self.d, chi_r_actual))
            chi_l = chi_r_actual

        # Build left and right environments
        self.L_envs = [None] * N  # L_envs[i]: left environment up to (not including) site i
        self.R_envs = [None] * N  # R_envs[i]: right environment from (not including) site i
        self._build_all_environments()

    def _build_all_environments(self):
        """Build all left and right environments from scratch."""
        N, d = self.N, self.d
        # Left environments: L_envs[0] = trivial scalar 1
        # Shape: (chi_left, chi_left)  -- the boundary scalar
        self.L_envs[0] = np.array([[[1.0]]])  # (1, 1, 1) = (bra, H, ket)
        # Right environments: R_envs[N-1] = trivial
        self.R_envs[N - 1] = np.array([[[1.0]]])

        # Build right envs from right to left
        for i in range(N - 2, -1, -1):
            self.R_envs[i] = self._contract_right(i + 1, self.R_envs[i + 1])

    def _contract_right(self, site, R):
        """Extend right environment to include site."""
        # This is a simplified contraction — for illustration
        A = self.tensors[site]  # (chi_l, d, chi_r)
        chi_l, d, chi_r = A.shape
        chi_R = R.shape[2]
        # New R: (chi_l, chi_l, chi_l) including local Hamiltonian terms
        # For the TFIM we track only the scalar overlap (simplified)
        R_new = np.einsum('iaj,jk,iak->ii', A.conj(), R[0], A).reshape(1, 1, 1)
        return R_new

    def sweep(self):
        """Perform one left-right sweep using full ED of the effective Hamiltonian."""
        N = self.N
        energies = []
        discarded_weights = []

        # Build full system Hamiltonian for the effective local update
        H_full = build_tfim(N, self.J, self.Gamma)

        # Single-site update via projecting onto the MPS subspace
        # (simplified: re-diagonalise in the effective basis)
        # For demonstration, we use the full Hamiltonian with a variational MPS ansatz
        evals, evecs = eigsh(H_full, k=1, which='SA')
        psi_target = evecs[:, 0]

        # Compress to MPS
        self.psi_approx = psi_target
        return evals[0]


# Demonstrate DMRG concept: variational energy vs bond dimension
print("Variational energy convergence with bond dimension χ:")
print(f"{'N':>5}  {'χ':>5}  {'E0/N (MPS)':>14}  {'E0/N (ED)':>14}  {'rel. error':>12}")
print("-" * 58)

for N in [10, 14, 18]:
    H = build_tfim(N, J=1.0, Gamma=1.0)
    evals_ed, evecs_ed = eigsh(H, k=1, which='SA')
    psi0 = evecs_ed[:, 0]
    E0_ed = evals_ed[0] / N

    for chi in [2, 4, 8, 16]:
        # Compress exact ground state to bond dimension chi
        C = psi0.copy().reshape(1, 2**N)
        chi_left = 1
        tensors = []
        for i in range(N - 1):
            C = C.reshape(chi_left * 2, 2**(N - 1 - i))
            U, s, Vt = np.linalg.svd(C, full_matrices=False)
            chi_r = min(len(s), chi)
            tensors.append(U[:, :chi_r].reshape(chi_left, 2, chi_r))
            chi_left = chi_r
            C = np.diag(s[:chi_r]) @ Vt[:chi_r, :]
        tensors.append(C.reshape(chi_left, 2, 1))

        # Reconstruct approximate wavefunction
        vec = tensors[0].reshape(2, -1)
        for i in range(1, N):
            A = tensors[i]
            vec = np.einsum('ac,cdr->adr', vec, A).reshape(-1, A.shape[2])
        psi_approx = vec.flatten()
        psi_approx /= np.linalg.norm(psi_approx)
        E_approx = float(psi_approx @ H @ psi_approx) / N

        rel_err = abs(E_approx - E0_ed) / abs(E0_ed)
        print(f"{N:>5}  {chi:>5}  {E_approx:>14.8f}  {E0_ed:>14.8f}  {rel_err:>12.2e}")
    print()

## 3. The Bond Dimension $\chi$

The bond dimension $\chi$ is the central parameter of DMRG:

| Method | Memory | Time | Accessible $N$ |
|---|---|---|---|
| ED (full) | $O(4^N)$ | $O(8^N)$ | $\lesssim 18$ |
| ED (Lanczos) | $O(2^N)$ | $O(k \cdot 2^N)$ | $\lesssim 40$ |
| DMRG ($\chi$ fixed) | $O(N\chi^2 d)$ | $O(N\chi^3 d^2)$ per sweep | $\sim 10^3$–$10^4$ |

Doubling $\chi$ increases cost by a factor of 8. For a gapped 1D system, $\chi = 100$ at $N = 1000$ is routine.

---

In [ ]:
# Show how required chi grows with system size for fixed accuracy
# at the critical point (hardest case) vs gapped phase
target_error = 1e-4
Ns = [8, 10, 12, 14, 16]
chi_needed_critical = []
chi_needed_gapped = []

for N in Ns:
    for g, chi_list in [(1.0, chi_needed_critical), (0.3, chi_needed_gapped)]:
        H = build_tfim(N, J=1.0, Gamma=g)
        _, evecs = eigsh(H, k=1, which='SA')
        psi0 = evecs[:, 0]
        E0_exact = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0] / N

        found_chi = None
        for chi in [1, 2, 4, 8, 16, 32, 64, 2**(N//2)]:
            C = psi0.copy().reshape(1, 2**N)
            chi_left = 1
            tensors = []
            for i in range(N - 1):
                C = C.reshape(chi_left * 2, 2**(N - 1 - i))
                U, s, Vt = np.linalg.svd(C, full_matrices=False)
                chi_r = min(len(s), chi)
                tensors.append(U[:, :chi_r].reshape(chi_left, 2, chi_r))
                chi_left = chi_r
                C = np.diag(s[:chi_r]) @ Vt[:chi_r, :]
            tensors.append(C.reshape(chi_left, 2, 1))
            vec = tensors[0].reshape(2, -1)
            for i in range(1, N):
                A = tensors[i]
                vec = np.einsum('ac,cdr->adr', vec, A).reshape(-1, A.shape[2])
            psi_approx = vec.flatten()
            psi_approx /= np.linalg.norm(psi_approx)
            E_approx = float(psi_approx @ H @ psi_approx) / N
            if abs(E_approx - E0_exact) / abs(E0_exact) < target_error:
                found_chi = chi
                break
        chi_list.append(found_chi if found_chi else 2**(N//2))

fig, ax = plt.subplots(figsize=(7, 5))
ax.semilogy(Ns, chi_needed_critical, 'r-o', label='Critical (Γ/J=1.0)')
ax.semilogy(Ns, chi_needed_gapped, 'b-o', label='Gapped (Γ/J=0.3)')
ax.set_xlabel('System size $N$')
ax.set_ylabel(r'$\chi$ needed for error $< 10^{-4}$')
ax.set_title('Bond dimension required vs system size')
ax.legend()
plt.tight_layout()
plt.show()

## 4. Truncation and the Discarded Weight

At each DMRG update step, after solving for the optimal tensor, we SVD and truncate to bond dimension $\chi$. The **discarded weight** $\epsilon = \sum_{\alpha > \chi} \lambda_\alpha^2$ is the key convergence diagnostic. Target $\epsilon < 10^{-8}$ for high-accuracy results; if it is large, increase $\chi$.

---

In [ ]:
# Discarded weight at each bond as a function of chi
N = 16
chi_values = [2, 4, 8, 16, 32]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, (g, label) in zip(axes, [(0.3, 'Ordered'), (1.0, 'Critical'), (3.0, 'Disordered')]):
    H = build_tfim(N, J=1.0, Gamma=g)
    _, evecs = eigsh(H, k=1, which='SA')
    psi0 = evecs[:, 0]

    for chi in chi_values:
        C = psi0.copy().reshape(1, 2**N)
        chi_left = 1
        disc_weights = []
        for i in range(N - 1):
            C = C.reshape(chi_left * 2, 2**(N - 1 - i))
            _, s, Vt = np.linalg.svd(C, full_matrices=False)
            chi_r = min(len(s), chi)
            disc_weights.append(np.sum(s[chi_r:]**2))
            chi_left = chi_r
            C = np.diag(s[:chi_r]) @ Vt[:chi_r, :]

        ax.semilogy(range(1, N), [w + 1e-16 for w in disc_weights], '-', label=f'χ={chi}')

    ax.set_xlabel('Bond index')
    ax.set_ylabel('Discarded weight')
    ax.set_title(label)
    ax.legend(fontsize=8)

plt.suptitle(f'Discarded weight per bond, N={N}', y=1.02)
plt.tight_layout()
plt.show()

## 5. Convergence and Sweep Strategy

DMRG convergence is assessed by monitoring the energy across sweeps:

- **Energy stabilisation:** change per sweep drops below a threshold (e.g. $10^{-10}$).
- **Declining discarded weight:** maximum discarded weight decreases as $\chi$ increases.
- **Consistent observables:** magnetization, entropy no longer change between sweeps.

**Warm-up:** initialise with a random MPS at small $\chi$, then increase progressively. In the ordered phase, add a small symmetry-breaking field during the first few sweeps to steer DMRG into the correct sector.

**Two-site vs single-site DMRG:** two-site updates allow the bond dimension to grow dynamically and are more robust against local minima, at higher cost per sweep. Most production codes use two-site DMRG for warm-up, then single-site for refinement.

**Boundary conditions:** DMRG converges faster with open boundary conditions (OBC). With PBC, the effective Hamiltonian involves long-range connections that are expensive to contract.

---

In [ ]:
# Simulate DMRG convergence: track energy as chi is progressively increased
# (mimicking a warm-up schedule without a full DMRG implementation)
N = 16
H = build_tfim(N, J=1.0, Gamma=1.0)  # critical point — hardest case
E0_exact = eigsh(H, k=1, which='SA', return_eigenvectors=False)[0]
_, evecs = eigsh(H, k=1, which='SA')
psi0 = evecs[:, 0]

chi_schedule = [1, 2, 4, 8, 16, 32, 64]
E_by_chi = []
discarded_by_chi = []

for chi in chi_schedule:
    C = psi0.copy().reshape(1, 2**N)
    chi_left = 1
    tensors, max_disc = [], 0.0
    for i in range(N - 1):
        C = C.reshape(chi_left * 2, 2**(N - 1 - i))
        U, s, Vt = np.linalg.svd(C, full_matrices=False)
        chi_r = min(len(s), chi)
        max_disc = max(max_disc, np.sum(s[chi_r:]**2))
        tensors.append(U[:, :chi_r].reshape(chi_left, 2, chi_r))
        chi_left = chi_r
        C = np.diag(s[:chi_r]) @ Vt[:chi_r, :]
    tensors.append(C.reshape(chi_left, 2, 1))
    vec = tensors[0].reshape(2, -1)
    for i in range(1, N):
        A = tensors[i]
        vec = np.einsum('ac,cdr->adr', vec, A).reshape(-1, A.shape[2])
    psi_approx = vec.flatten() / np.linalg.norm(vec.flatten())
    E_by_chi.append(float(psi_approx @ H @ psi_approx))
    discarded_by_chi.append(max_disc)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].semilogx(chi_schedule, [(e - E0_exact) / abs(E0_exact) for e in E_by_chi], 'b-o')
axes[0].axhline(0, color='red', linestyle='--', alpha=0.5)
axes[0].set_xlabel(r'Bond dimension $\chi$')
axes[0].set_ylabel(r'$(E - E_0) / |E_0|$')
axes[0].set_title('Energy convergence with χ')

axes[1].loglog(chi_schedule, [d + 1e-16 for d in discarded_by_chi], 'r-o')
axes[1].set_xlabel(r'Bond dimension $\chi$')
axes[1].set_ylabel('Max discarded weight')
axes[1].set_title('Discarded weight vs χ')

plt.suptitle(f'DMRG convergence monitor — N={N}, critical point', y=1.02)
plt.tight_layout()
plt.show()
print(f"Exact E0 = {E0_exact:.8f}")
print(f"MPS (χ={chi_schedule[-1]}) E0 = {E_by_chi[-1]:.8f}")

## 6. What DMRG Can and Cannot Do

**Excels at:**
- Ground states of 1D gapped systems: essentially exact for modest $\chi$.
- Long chains: $N \sim 10^3$–$10^4$ sites routine for gapped systems.
- Open boundary conditions: faster convergence than PBC.

**Struggles with:**
- 2D systems: entanglement grows as $S \sim W$, requiring $\chi \sim e^W$ — exponential in width.
- Critical systems: requires large $\chi$ growing with system size.
- Excited states and dynamics: extensions (TEBD, TDVP) exist but are more challenging.

Lecture 10 puts these capabilities into practice using TeNPy.

---